In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# SHARED PROTEIN INTERACTION

# BIOGRID Shared PPI Feature
Downloaded from: https://downloads.thebiogrid.org/BioGRID/Release-Archive/BIOGRID-4.4.241/

In [2]:
def process_biogrid_ppi_data(data):
    # Filter interactions down to physical interactions between two human proteins
    hsapien = 9606
    biogrid_ppi = data[(data['Organism ID Interactor A']==hsapien) & (data['Organism ID Interactor B']==hsapien) &
                       (data['Experimental System Type']=='physical')]
    # Filter to columns of interest + rename
    biogrid_ppi = biogrid_ppi[['Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'Experimental System']]
    biogrid_ppi = biogrid_ppi.rename(columns={'Entrez Gene Interactor A':'A1_entrez', 'Entrez Gene Interactor B':'A2_entrez',
                                              'Experimental System':'experimental_system'})
    biogrid_ppi = biogrid_ppi.drop_duplicates().reset_index(drop=True).astype({'A1_entrez':'int', 'A2_entrez':'int'})
    
    # Filter out self interactions
    biogrid_ppi = biogrid_ppi[biogrid_ppi.A1_entrez != biogrid_ppi.A2_entrez].reset_index(drop=True)
    print('N interactions:', biogrid_ppi.shape[0])
    
    # Sort interacting genes - to find unique interaction pairs
    biogrid_unique = pd.DataFrame(np.sort(biogrid_ppi[['A1_entrez','A2_entrez']], axis=1), 
                                  columns=['A1_entrez', 'A2_entrez']).drop_duplicates()
    assert(biogrid_unique.shape[0] <= biogrid_ppi.shape[0])

    print('N interactions after sorting for unique pairs:', biogrid_unique.shape[0])
    print('N genes in interaction map:', pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).nunique())
    
    return biogrid_unique

In [ ]:
biogrid_raw = pd.read_csv('input_data/BIOGRID/BIOGRID-ALL-4.4.241.tab3.txt', sep='\t',  low_memory=False)
biogrid_raw[:1]
biogrid_unique = process_biogrid_ppi_data(biogrid_raw)
biogrid_genes = pd.concat([biogrid_unique.A1_entrez, biogrid_unique.A2_entrez]).unique()
biogrid_unique

N interactions: 993165
N interactions after sorting for unique pairs: 862452
N genes in interaction map: 19995


,A1_entrez,A2_entrez
0,2318,6416
1,88,84665
2,90,2339
3,2624,5371
4,6118,6774
...,...,...
993159,4085,26528
993160,4085,10521
993161,4085,10432
993162,9817,29843


In [9]:
## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df

In [10]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

biomarker_query = main_file_csv[['entrez_id_biomarker','entrez_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'entrez_id_biomarker': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
biomarker_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,biomarker_query)
biomarker_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_biomarker_query.csv',index=False)
biomarker_query_fet_shared_ppi

N. gene pairs that interact: 5056
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5728.0,5296.0,True,"{387, 6156, 2065, 4628, 3861, 8471, 408, 31, 4...",813,34,0.041820,14.036345
1,5728.0,3636.0,False,"{8451, 10376, 3850, 10381, 10383, 2065, 7316, ...",798,37,0.046366,17.368386
2,5728.0,7249.0,False,"{801, 5347, 867, 2885, 7879, 207, 2099, 5747, ...",770,14,0.018182,3.468631
3,5728.0,4771.0,False,"{4609, 5250, 6147, 8450, 8451, 8454, 6279, 628...",895,68,0.075978,35.100297
4,5728.0,63932.0,False,"{3329, 4609, 6160, 55709, 10528, 4000, 4001, 7...",689,19,0.027576,11.767982
...,...,...,...,...,...,...,...,...
144075,4893.0,440590.0,False,{},692,0,0.000000,0.000000
144076,4893.0,79699.0,False,"{1994, 440730}",763,2,0.002621,-0.112343
144077,4893.0,7791.0,False,"{5127, 11017, 9738, 10768, 440730, 57498, 1091...",960,29,0.030208,5.952641
144078,4893.0,23140.0,False,"{6117, 6118, 6119, 53834, 10618, 10551, 440730...",793,8,0.010088,1.097225


In [11]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target1_query = main_file_csv[['entrez_id_target1','entrez_id_query']]
target1_query = target1_query.rename(columns={
    'entrez_id_target1': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target1_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,target1_query)
target1_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target1_query.csv',index=False)
target1_query_fet_shared_ppi

N. gene pairs that interact: 2919
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5291.0,5296.0,True,"{84868, 3845, 6790, 7048, 2064, 2065, 8471, 10...",262,30,0.114504,38.971872
1,5291.0,3636.0,False,"{867, 1956, 2885, 5295, 2064, 2065, 4914, 5296...",271,9,0.033210,7.069189
2,5291.0,7249.0,False,"{867, 2885, 328, 5868, 8974, 207, 4914, 10870,...",220,9,0.040909,8.158703
3,5291.0,4771.0,False,"{1956, 3845, 6790, 7175, 328, 11113, 9928, 391...",396,12,0.030303,8.312495
4,5291.0,63932.0,False,{328},152,1,0.006579,0.594197
...,...,...,...,...,...,...,...,...
144075,5604.0,440590.0,False,{},262,0,0.000000,0.000000
144076,5604.0,79699.0,False,"{1994, 440730}",333,2,0.006006,0.511541
144077,5604.0,7791.0,False,"{55832, 440730, 7706, 1051, 10413, 699, 57664,...",534,25,0.046816,12.893010
144078,5604.0,23140.0,False,"{57664, 6117, 6790, 6118, 6119, 5071, 440730}",364,7,0.019231,3.010342


In [12]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target2_query = main_file_csv[['entrez_id_target2','entrez_id_query']]
target2_query = target2_query.rename(columns={
    'entrez_id_target2': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target2_query_fet_shared_ppi = compute_ppi_summary_for_pairs(biogrid_unique,target2_query)
target2_query_fet_shared_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target2_query.csv',index=False)
target2_query_fet_shared_ppi

N. gene pairs that interact: 913
N paralog pairs w/ 1+ interactor (A1 and/or A2): 141279
N genes involded in interactions: 19995


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,NaN,5296.0,False,{},217,0,0.000000,0.000000
1,NaN,3636.0,False,{},203,0,0.000000,0.000000
2,NaN,7249.0,False,{},152,0,0.000000,0.000000
3,NaN,4771.0,False,{},331,0,0.000000,0.000000
4,NaN,63932.0,False,{},76,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...
144075,5605.0,440590.0,False,{},277,0,0.000000,0.000000
144076,5605.0,79699.0,False,"{440730, 10987}",348,2,0.005747,0.477181
144077,5605.0,7791.0,False,"{440730, 7706, 23586, 10788, 55212, 9531, 2108...",554,20,0.036101,8.240295
144078,5605.0,23140.0,False,"{57664, 6117, 6118, 6119, 5071, 440730}",380,6,0.015789,2.195086


# STRING Shared PPI Feature

In [ ]:
hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('input_data/STRING/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('input_data/STRING/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]
string = string[string['combined_score'] >= 400].sort_values(by='combined_score', ascending=False)

string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)

# === HGNC lookup
gene_lookup = {}
def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)
    entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
    syms = []
    s = normalize_symbol(entry.get('symbol'))
    if s: syms.append(s)
    if pd.notnull(entry.get('prev_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
    if pd.notnull(entry.get('alias_symbol')):
        syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
    for sym in syms:
        if sym and sym not in gene_lookup:
            gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)

def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

string['hgnc_id_1'], string['ensembl_gene_id_1'], string['entrez_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'], string['entrez_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))

string


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_91821/3396990575.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2,hgnc_id_1,ensembl_gene_id_1,entrez_id_1,hgnc_id_2,ensembl_gene_id_2,entrez_id_2
0,ENSP00000259469,ENSP00000481646,0,0,0,959,989,0,514,999,RPL35,RPS10,HGNC:10344,ENSG00000136942,11224.0,HGNC:10383,ENSG00000124614,6204.0
1,ENSP00000229214,ENSP00000454836,0,0,0,759,994,360,243,999,KRR1,TBL3,HGNC:5176,ENSG00000111615,11103.0,HGNC:11587,ENSG00000183751,10607.0
2,ENSP00000260762,ENSP00000253861,0,0,0,60,978,800,999,999,EXOC6,EXOC4,HGNC:23196,ENSG00000138190,54536.0,HGNC:30389,ENSG00000131558,60412.0
3,ENSP00000294189,ENSP00000389103,0,0,0,991,994,0,502,999,RPL29,RPL23A,HGNC:10331,ENSG00000162244,6159.0,HGNC:10317,ENSG00000198242,6147.0
4,ENSP00000260762,ENSP00000370695,0,0,0,129,921,800,999,999,EXOC6,EXOC1,HGNC:23196,ENSG00000138190,54536.0,HGNC:30380,ENSG00000090989,55763.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1858939,ENSP00000311005,ENSP00000498019,0,0,0,0,400,0,0,400,MAPK7,RPS17,HGNC:6880,ENSG00000166484,5598.0,HGNC:10397,ENSG00000182774,6218.0
1858940,ENSP00000398495,ENSP00000248553,0,0,0,42,0,0,400,400,DIABLO-2,HSPB1,NaN,NaN,NA,HGNC:5246,ENSG00000106211,3315.0
1858941,ENSP00000357588,ENSP00000478070,0,0,0,400,0,0,0,400,CALHM5,CWC25,HGNC:21568,ENSG00000178033,254228.0,HGNC:25989,ENSG00000273559,54883.0
1858942,ENSP00000375622,ENSP00000395294,0,0,0,0,0,0,400,400,LAIR1,CD70,HGNC:6477,ENSG00000167613,3903.0,HGNC:11937,ENSG00000125726,970.0


In [16]:
string_unique = (
    string[["entrez_id_1", "entrez_id_2"]]
    .rename(columns={"entrez_id_1": "A1_entrez", "entrez_id_2": "A2_entrez"})
    .drop_duplicates()
    .reset_index(drop=True)
)

# Convert safely, making "NA" into NaN first
string_unique["A1_entrez"] = pd.to_numeric(string_unique["A1_entrez"], errors="coerce").astype("Int64")
string_unique["A2_entrez"] = pd.to_numeric(string_unique["A2_entrez"], errors="coerce").astype("Int64")

print('N interactions:', string_unique.shape[0])

# Drop rows with missing values before sorting
string_unique = string_unique.dropna(subset=["A1_entrez", "A2_entrez"])

# Sort interacting genes - to find unique interaction pairs
string_unique = (
    pd.DataFrame(np.sort(string_unique[['A1_entrez','A2_entrez']], axis=1), 
                 columns=['A1_entrez', 'A2_entrez'])
    .drop_duplicates()
)
string_unique = string_unique[string_unique["A1_entrez"] != string_unique["A2_entrez"]]

print('N interactions after sorting for unique pairs:', string_unique.shape[0])
print('N genes in interaction map:', pd.concat([string_unique.A1_entrez, string_unique.A2_entrez]).nunique())

string_unique


N interactions: 1840450
N interactions after sorting for unique pairs: 915312
N genes in interaction map: 19004


,A1_entrez,A2_entrez
0,6204,11224
1,10607,11103
2,54536,60412
3,6147,6159
4,54536,55763
...,...,...
1830039,3439,57864
1830050,27161,266743
1830175,3646,79752
1830325,7979,57804


In [21]:
## Calculate shared and total protein-protein interactions for all gene pairs
def compute_ppi_summary_for_pairs(ppi, gene_pairs):
    
    # Make ppi df symmetrical for merging with paralog pairs (which are the unique list)
    ppi_symmetric = pd.concat([ppi, ppi.rename(columns={'A1_entrez':'A2_entrez','A2_entrez':'A1_entrez'})])
    ppi_symmetric = ppi_symmetric.reset_index(drop=True)
    assert(ppi_symmetric.shape[0] == ppi.shape[0]*2)
    
    # Merge with paralog pairs - label interaction if paralog pair is found in PPI df
    gene_ppi = pd.merge(gene_pairs[['A1_entrez','A2_entrez']], ppi_symmetric, how='left', indicator='interact')
    gene_ppi.interact = gene_ppi.interact=='both'
    assert(gene_ppi.shape[0] == gene_pairs.shape[0])
    print('N. gene pairs that interact:', sum(gene_ppi.interact))
    
    # Use symmetric version of ppi table to get set of all interactors for each gene
    ppi_per_gene = ppi_symmetric.groupby('A1_entrez').agg({'A2_entrez':set}).reset_index()\
                                .rename(columns={'A1_entrez':'gene', 'A2_entrez':'ppi'})
    assert(ppi_per_gene.shape[0] == pd.concat([ppi.A1_entrez, ppi.A2_entrez]).nunique())

    # Merge ppi_per_gene with each A1 and A2 in all pairs
    # Note: pairs can have shared interactors even if there is no evidence they interact themselves
    df = pd.merge(gene_ppi, ppi_per_gene.rename(columns={'gene':'A1_entrez','ppi':'A1_ppi'}), how='left')
    df = pd.merge(df, ppi_per_gene.rename(columns={'gene':'A2_entrez','ppi':'A2_ppi'}), how='left')
    assert(df.shape[0] == gene_pairs.shape[0])
    print('N paralog pairs w/ 1+ interactor (A1 and/or A2):', df[(~df.A1_ppi.isna()) | (~df.A2_ppi.isna())].shape[0])
    
    # Fill NaNs with empty sets
    df['A1_ppi'] = df['A1_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    df['A2_ppi'] = df['A2_ppi'].apply(lambda d: d if not pd.isnull(d) else set())
    
    # Remove A2 gene in the set of interactors for A1 gene (and vice versa)
    # Don't want to include these in union for other calculations
    df.A1_ppi = df.apply(lambda x: x.A1_ppi - {x.A2_entrez}, axis=1)
    df.A2_ppi = df.apply(lambda x: x.A2_ppi - {x.A1_entrez}, axis=1)

    # Calculate total num interactors + shared interactors
    df['n_A1_ppi'] = df.apply(lambda x: len(x.A1_ppi), axis=1)
    df['n_A2_ppi'] = df.apply(lambda x: len(x.A2_ppi), axis=1)
    df['shared_ppi'] = df.apply(lambda x: x.A1_ppi.intersection(x.A2_ppi), axis=1)
    df['n_total_ppi'] = df.apply(lambda x: len(x.A1_ppi.union(x.A2_ppi)), axis=1)
    df['n_shared_ppi'] = df.apply(lambda x: len(x.shared_ppi), axis=1)

    # Calculate jaccard index for shared interactors
    def calc_jaccard_index(x):
        if x.n_shared_ppi == 0: return 0
        return x.n_shared_ppi / ((x.n_A1_ppi + x.n_A2_ppi) - x.n_shared_ppi)

    df['shared_ppi_jaccard_idx'] = df.apply(calc_jaccard_index, axis=1)

    # Calculate FET for overlap of interactors, N = all genes involved in interactions
    N = len(pd.concat([ppi.A1_entrez, ppi.A2_entrez]).unique())
    print('N genes involded in interactions:', N)
    assert(ppi_per_gene.shape[0] == N)
    # ctab:   | A2      | Not A2
    #      A1 | shared  | A1 only
    #  Not A1 | A2 only | N - union(A1, A2)
    def calc_fet_shared_ppi(x):
        ctab = pd.DataFrame({'A2': [x.n_shared_ppi, x.n_A2_ppi - x.n_shared_ppi],
                            'NA2': [x.n_A1_ppi - x.n_shared_ppi, N - x.n_total_ppi]}, index=['A1', 'NA1'])
        (OR, pval) = stats.fisher_exact(ctab)
        if pval==0: # Use smallest float64 number, to apply log10
            pval = np.nextafter(0, 1)
        log_pval = (-np.log10(pval)) if pval != 1 else 0
        log_pval = -log_pval if OR < 1 else log_pval # return negative pval if signif depletion in overlap
        return log_pval

    df['fet_ppi_overlap'] = df.apply(calc_fet_shared_ppi, axis=1)
    
    df = df.drop(columns=['A1_ppi','A2_ppi','n_A1_ppi','n_A2_ppi'])
    
    return df

In [22]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

biomarker_query = main_file_csv[['entrez_id_biomarker','entrez_id_query']]
biomarker_query = biomarker_query.rename(columns={
    'entrez_id_biomarker': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
biomarker_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,biomarker_query)
biomarker_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_biomarker_query_string.csv',index=False)
biomarker_query_fet_shared_string_ppi

N. gene pairs that interact: 6007
N paralog pairs w/ 1+ interactor (A1 and/or A2): 144080
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5728.0,5296.0,True,"{55300, 4615, 8202, 3082, 2064, 2065, 2066, 71...",1298,178,0.137134,114.152228
1,5728.0,3636.0,True,"{200576, 26499, 3716, 3717, 3718, 387, 4233, 5...",1152,57,0.049479,34.068956
2,5728.0,7249.0,True,"{4609, 1026, 1027, 1029, 1019, 2064, 2065, 309...",1202,181,0.150582,144.356054
3,5728.0,4771.0,True,"{6657, 4609, 1029, 1030, 3082, 3084, 2064, 206...",1158,95,0.082038,68.669701
4,5728.0,63932.0,False,"{1457, 283106, 1459}",1142,3,0.002627,-0.097172
...,...,...,...,...,...,...,...,...
144075,4893.0,440590.0,False,{},621,0,0.000000,0.000000
144076,4893.0,79699.0,False,{},626,0,0.000000,-0.202768
144077,4893.0,7791.0,False,"{4609, 391, 3725, 409, 7837, 2335, 1956, 1969,...",699,21,0.030043,9.351460
144078,4893.0,23140.0,False,{},624,0,0.000000,-0.203459


In [23]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target1_query = main_file_csv[['entrez_id_target1','entrez_id_query']]
target1_query = target1_query.rename(columns={
    'entrez_id_target1': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target1_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,target1_query)
target1_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target1_query_string.csv',index=False)
target1_query_fet_shared_string_ppi

N. gene pairs that interact: 4529
N paralog pairs w/ 1+ interactor (A1 and/or A2): 143870
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,5291.0,5296.0,True,"{118788, 51206, 8202, 2064, 2065, 2066, 7189, ...",470,281,0.597872,323.306215
1,5291.0,3636.0,True,"{146433, 26499, 3716, 3717, 3718, 387, 4233, 7...",415,69,0.166265,83.017283
2,5291.0,7249.0,True,"{30849, 1026, 387, 2308, 1029, 2309, 5894, 384...",583,75,0.128645,62.190596
3,5291.0,4771.0,False,"{387, 1029, 3845, 4233, 2064, 2065, 2066, 5781...",487,43,0.088296,34.650616
4,5291.0,63932.0,False,{},420,0,0.000000,-0.188936
...,...,...,...,...,...,...,...,...
144075,5604.0,440590.0,False,{},414,0,0.000000,0.000000
144076,5604.0,79699.0,False,{},419,0,0.000000,0.000000
144077,5604.0,7791.0,False,"{4609, 387, 3725, 6416, 409, 7837, 2335, 28964...",489,24,0.049080,16.005291
144078,5604.0,23140.0,False,{},417,0,0.000000,0.000000


In [24]:
main_file_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv',low_memory=False)

target2_query = main_file_csv[['entrez_id_target2','entrez_id_query']]
target2_query = target2_query.rename(columns={
    'entrez_id_target2': 'A1_entrez',
    'entrez_id_query' : 'A2_entrez'
})
target2_query_fet_shared_string_ppi = compute_ppi_summary_for_pairs(string_unique,target2_query)
target2_query_fet_shared_string_ppi.to_csv('input_data/3_ML_outputs/feature_output/fet_ppi_overlap_target2_query_string.csv',index=False)
target2_query_fet_shared_string_ppi

N. gene pairs that interact: 932
N paralog pairs w/ 1+ interactor (A1 and/or A2): 143396
N genes involded in interactions: 19004


,A1_entrez,A2_entrez,interact,shared_ppi,n_total_ppi,n_shared_ppi,shared_ppi_jaccard_idx,fet_ppi_overlap
0,NaN,5296.0,False,{},407,0,0.000000,0.000000
1,NaN,3636.0,False,{},140,0,0.000000,0.000000
2,NaN,7249.0,False,{},314,0,0.000000,0.000000
3,NaN,4771.0,False,{},184,0,0.000000,0.000000
4,NaN,63932.0,False,{},74,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...
144075,5605.0,440590.0,False,{},213,0,0.000000,0.000000
144076,5605.0,79699.0,False,{},218,0,0.000000,0.000000
144077,5605.0,7791.0,False,"{387, 1956, 998, 3690, 3725, 207, 54518, 7414,...",298,14,0.046980,10.589875
144078,5605.0,23140.0,False,{},216,0,0.000000,0.000000


# STRING Interaction Score

In [ ]:
hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('input_data/STRING/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('input_data/STRING/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2181448581.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


In [ ]:
string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)


string


,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2
0,ENSP00000000233,ENSP00000356607,0,0,0,45,134,0,81,173,ARF5,RALGPS2
1,ENSP00000000233,ENSP00000427567,0,0,0,0,128,0,70,154,ARF5,FHDC1
2,ENSP00000000233,ENSP00000253413,0,0,0,118,49,0,69,151,ARF5,ATP6V1E1
3,ENSP00000000233,ENSP00000493357,0,0,0,56,53,0,457,471,ARF5,CYTH2
4,ENSP00000000233,ENSP00000324127,0,0,0,0,46,0,197,201,ARF5,PSD3
...,...,...,...,...,...,...,...,...,...,...,...,...
13715399,ENSP00000501317,ENSP00000475489,0,0,0,60,99,0,126,195,RFX7,MPHOSPH9
13715400,ENSP00000501317,ENSP00000370447,0,0,0,55,111,0,79,158,RFX7,VCX
13715401,ENSP00000501317,ENSP00000312272,0,0,0,0,0,0,227,226,RFX7,YPEL2
13715402,ENSP00000501317,ENSP00000402092,0,0,0,0,67,0,146,169,RFX7,SAMD3


In [ ]:
# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))


# Ensure all entries are treated as strings
string['gene_symbol_1'] = string['gene_symbol_1'].astype(str)
string['gene_symbol_2'] = string['gene_symbol_2'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
string['hgnc_id_1'], string['ensembl_gene_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))
string

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2,hgnc_id_1,ensembl_gene_id_1,hgnc_id_2,ensembl_gene_id_2
0,ENSP00000000233,ENSP00000356607,0,0,0,45,134,0,81,173,ARF5,RALGPS2,HGNC:658,ENSG00000004059,HGNC:30279,ENSG00000116191
1,ENSP00000000233,ENSP00000427567,0,0,0,0,128,0,70,154,ARF5,FHDC1,HGNC:658,ENSG00000004059,HGNC:29363,ENSG00000137460
2,ENSP00000000233,ENSP00000253413,0,0,0,118,49,0,69,151,ARF5,ATP6V1E1,HGNC:658,ENSG00000004059,HGNC:857,ENSG00000131100
3,ENSP00000000233,ENSP00000493357,0,0,0,56,53,0,457,471,ARF5,CYTH2,HGNC:658,ENSG00000004059,HGNC:9502,ENSG00000105443
4,ENSP00000000233,ENSP00000324127,0,0,0,0,46,0,197,201,ARF5,PSD3,HGNC:658,ENSG00000004059,HGNC:19093,ENSG00000156011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13715399,ENSP00000501317,ENSP00000475489,0,0,0,60,99,0,126,195,RFX7,MPHOSPH9,HGNC:25777,ENSG00000181827,HGNC:7215,ENSG00000051825
13715400,ENSP00000501317,ENSP00000370447,0,0,0,55,111,0,79,158,RFX7,VCX,HGNC:25777,ENSG00000181827,HGNC:12667,ENSG00000182583
13715401,ENSP00000501317,ENSP00000312272,0,0,0,0,0,0,227,226,RFX7,YPEL2,HGNC:25777,ENSG00000181827,HGNC:18326,ENSG00000175155
13715402,ENSP00000501317,ENSP00000402092,0,0,0,0,67,0,146,169,RFX7,SAMD3,HGNC:25777,ENSG00000181827,HGNC:21574,ENSG00000164483


In [ ]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
main_csv

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1094071010.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}
# Also add the reverse pairs since interactions might not be directional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

def get_combined_score(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
    pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # Reverse order

    return string_dict.get(pair1) or string_dict.get(pair2)  # Check both ways

# Apply function to find the interaction score
main_csv['StringInteractionWithBiomarker'] = main_csv.apply(get_combined_score, axis=1)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,985.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,960.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,889.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,606.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
string_biomarker_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_biomarker','StringInteractionWithBiomarker']]
string_biomarker_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_biomarker.csv',index=False)

In [ ]:
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}
# Also add the reverse pairs since interactions might not be directional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

def get_combined_score(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
    pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # Reverse order

    return string_dict.get(pair1) or string_dict.get(pair2)  # Check both ways

# Apply function to find the interaction score
main_csv['StringInteractionWithTarget'] = main_csv.apply(get_combined_score, axis=1)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,985.0,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,960.0,964.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,889.0,510.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,606.0,183.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
string_target1_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target1','StringInteractionWithTarget']]
string_target1_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_target1.csv',index=False)

In [ ]:
import pandas as pd

# Build the interaction dictionary from the STRING DataFrame
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}

# Add reverse pairs to make the dictionary bidirectional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

# Define function to get combined interaction score
def get_combined_score(row):
    target2 = row['ensembl_gene_id_target2']

    # Return 0 if target2 is missing or empty
    if pd.isna(target2) or target2 == '':
        return 0

    pair1 = (row['ensembl_gene_id_query'], target2)
    pair2 = (target2, row['ensembl_gene_id_query'])  # Reverse order

    # Return the score if found, else 0
    return string_dict.get(pair1) or string_dict.get(pair2) or 0

# Apply the function to the main DataFrame
main_csv['StringInteractionWithTarget2'] = main_csv.apply(get_combined_score, axis=1)

# Display the updated DataFrame
main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,StringInteractionWithTarget2
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,985.0,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,960.0,964.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,889.0,510.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,606.0,183.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
string_target2_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target2','StringInteractionWithTarget2']]
string_target2_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_target2.csv',index=False)

# BIOGRID Interaction Score

In [ ]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1717061817.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


In [ ]:
hgnc = pd.read_csv('input_data/HGNC/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

biogrid = pd.read_csv('input_data/BIOGRID/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]
biogrid

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2034133230.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2034133230.py:5: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,TREMBL Accessions Interactor B,REFSEQ Accessions Interactor B,Ontology Term IDs,Ontology Term Names,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,Q59H94,NP_001120959|NP_001449,-,-,-,-,-,-,Homo sapiens,Homo sapiens
1,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,NP_150250|NP_150253|NP_150252|NP_150247|NP_150...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
2,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,NP_644805|NP_003141|NP_001356447|NP_001356443|...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
3,586,375,23163,106870,116775,-,-,ARF1,GGA3,-,...,B7Z456|A8K6M0,NP_619525|NP_001278571|NP_001278570|NP_0011661...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
4,612,377,23647,106872,117174,-,-,ARF3,ARFIP2,-,...,B4DUZ3|B4DXH2|A0A087X1E4,NP_036534|NP_001229785|NP_001229784|NP_001229783,-,-,-,-,-,-,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472380,3638570,5566,55755,111553,120873,-,RP11-412P1.1,PRKACA,CDK5RAP2,PKACA,...,Q7Z3M0|B3KVI2|B9EG74,NP_060719|NP_001258968|NP_001011649,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472381,3638571,5566,9659,111553,115017,-,RP4-791M13.2,PRKACA,PDE4DIP,PKACA,...,A0A087WYE4|A0A087WVF8,NP_055459|NP_001337451|NP_001185763|NP_0011857...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472382,3638572,5566,113251,111553,125238,-,PP13296,PRKACA,LARP4,PKACA,...,Q6P4E2|Q96J85|Q8TBL5,NP_001339245|NP_001339244|NP_001339247|NP_0013...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472383,3638573,5566,2534,111553,108810,-,RP1-66H14.1,PRKACA,FYN,PKACA,...,-,NP_002028|NP_001357458|NP_694593|NP_694592,-,-,-,-,-,-,Homo sapiens,Homo sapiens


In [ ]:

# Sort values in each row and create a new column containing sorted combinations
biogrid['SortedInteractors'] = biogrid.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)

# Drop duplicates based on the new column
biogrid.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)

# Drop the auxiliary column
biogrid.drop(columns='SortedInteractors', inplace=True)

biogrid= biogrid.reset_index(drop=True)

# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan))



# Ensure all entries are treated as strings
biogrid['Official Symbol Interactor A'] = biogrid['Official Symbol Interactor A'].astype(str)
biogrid['Official Symbol Interactor B'] = biogrid['Official Symbol Interactor B'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
biogrid['hgnc_id_1'], biogrid['ensembl_gene_id_1'] = zip(*biogrid['Official Symbol Interactor A'].apply(lookup_gene_info))
biogrid['hgnc_id_2'], biogrid['ensembl_gene_id_2'] = zip(*biogrid['Official Symbol Interactor B'].apply(lookup_gene_info))

biogrid

,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B,hgnc_id_1,ensembl_gene_id_1,hgnc_id_2,ensembl_gene_id_2
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:6844,ENSG00000065559,HGNC:3756,ENSG00000128591
1,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:4171,ENSG00000179348,HGNC:9113,ENSG00000140464
2,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:10290,ENSG00000117748,HGNC:11364,ENSG00000168610
3,586,375,23163,106870,116775,-,-,ARF1,GGA3,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:652,ENSG00000143761,HGNC:17079,ENSG00000125447
4,612,377,23647,106872,117174,-,-,ARF3,ARFIP2,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:654,ENSG00000134287,HGNC:17160,ENSG00000132254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84513,3629930,582,9867,107058,115200,-,-,BBS1,PJA2,BBS2L2,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:966,ENSG00000174483,HGNC:17481,ENSG00000198961
84514,3629941,8975,7291,114465,113142,-,-,USP13,TWIST1,ISOT3|IsoT-3,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:12611,ENSG00000058056,HGNC:12428,ENSG00000122691
84515,3629949,5562,1317,111549,107712,-,RP11-110L1.1,PRKAA1,SLC31A1,AMPK|AMPKa1,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9376,ENSG00000132356,HGNC:11016,ENSG00000136868
84516,3638506,5566,2534,111553,108810,-,RP1-66H14.1,PRKACA,FYN,PKACA,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9380,ENSG00000072062,HGNC:4037,ENSG00000010810


In [ ]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
    pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryBiomarker'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
biogrid_query_biomarker = main_csv[['ensembl_gene_id_query','ensembl_gene_id_biomarker','BIOGRIDPhysicalInteractionQueryBiomarker']]
biogrid_query_biomarker.to_csv('input_data/3_ML_outputs/feature_output/biogrid_biomarker_query.csv',index=False)
biogrid_query_biomarker

,ensembl_gene_id_query,ensembl_gene_id_biomarker,BIOGRIDPhysicalInteractionQueryBiomarker
0,ENSG00000105647,ENSG00000171862,1
1,ENSG00000165458,ENSG00000171862,0
2,ENSG00000103197,ENSG00000171862,0
3,ENSG00000186575,ENSG00000171862,0
4,ENSG00000018610,ENSG00000171862,0
...,...,...,...
144075,ENSG00000203995,ENSG00000213281,0
144076,ENSG00000162378,ENSG00000213281,0
144077,ENSG00000159840,ENSG00000213281,0
144078,ENSG00000074755,ENSG00000213281,0


In [ ]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
    pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryTarget1'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [ ]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target2'])
    pair2 = (row['ensembl_gene_id_target2'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryTarget2'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1,BIOGRIDPhysicalInteractionQueryTarget2
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [ ]:
main_csv['BIOGRIDPhysicalInteractionQueryTarget'] = (main_csv['BIOGRIDPhysicalInteractionQueryTarget1'] | main_csv['BIOGRIDPhysicalInteractionQueryTarget2']).astype(int)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1,BIOGRIDPhysicalInteractionQueryTarget2,BIOGRIDPhysicalInteractionQueryTarget
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0


In [ ]:
biogrid_query_target = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target1','BIOGRIDPhysicalInteractionQueryTarget']]
biogrid_query_target.to_csv('input_data/3_ML_outputs/feature_output/biogrid_target_query.csv',index=False)
biogrid_query_target

,ensembl_gene_id_query,ensembl_gene_id_target1,BIOGRIDPhysicalInteractionQueryTarget
0,ENSG00000105647,ENSG00000051382,1
1,ENSG00000165458,ENSG00000051382,0
2,ENSG00000103197,ENSG00000051382,0
3,ENSG00000186575,ENSG00000051382,0
4,ENSG00000018610,ENSG00000051382,0
...,...,...,...
144075,ENSG00000203995,ENSG00000169032,0
144076,ENSG00000162378,ENSG00000169032,0
144077,ENSG00000159840,ENSG00000169032,0
144078,ENSG00000074755,ENSG00000169032,0


# BIOMARKER TSG or Oncogenes

In [ ]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
main_csv

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1094071010.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
main_csv['Biomarker'].unique()

array(['PTEN', 'NRAS', 'BRAF', 'ARID1A', 'KRAS', 'BRCA1'], dtype=object)

# Biomarker TSG or oncogene

In [ ]:
cancer_gene_census = pd.read_csv('input_data/Cancer_Gene_Census/Cancer_gene_census_data.csv')
cancer_gene_census

,Gene Symbol,Name,Entrez GeneId,Genome Location,Tier,Hallmark,Chr Band,Somatic,Germline,Tumour Types(Somatic),Tumour Types(Germline),Cancer Syndrome,Tissue Type,Molecular Genetics,Role in Cancer,Mutation Types,Translocation Partner,Other Germline Mut,Other Syndrome,Synonyms
0,A1CF,APOBEC1 complementation factor,29974.0,10:50799421-50885675,2,NaN,10q11.23,yes,NaN,melanoma,NaN,NaN,E,NaN,oncogene,Mis,NaN,NaN,NaN,"ACF,ACF64,ACF65,APOBEC1CF,ASP,CCDS73133.1,ENSG..."
1,ABI1,abl-interactor 1,10006.0,10:26746593-26860935,1,Yes,10p12.1,yes,NaN,AML,NaN,NaN,L,Dom,"TSG, fusion",T,KMT2A,NaN,NaN,"ABI-1,CCDS7150.1,E3B1,ENSG00000136754.17,NM_00..."
2,ABL1,v-abl Abelson murine leukemia viral oncogene h...,25.0,9:130713946-130885683,1,Yes,9q34.12,yes,NaN,"CML, ALL, T-ALL",NaN,NaN,L,Dom,"oncogene, fusion","T, Mis","BCR, ETV6, NUP214",NaN,NaN,"ABL,CCDS35165.1,ENSG00000097007.17,JTK7,NM_007..."
3,ABL2,"c-abl oncogene 2, non-receptor tyrosine kinase",27.0,1:179099327-179229601,1,NaN,1q25.2,yes,NaN,AML,NaN,NaN,L,Dom,"oncogene, fusion",T,ETV6,NaN,NaN,"ABLL,ARG,CCDS30947.1,ENSG00000143322.19,NM_007..."
4,ACKR3,atypical chemokine receptor 3,57007.0,2:236569641-236582358,1,Yes,2q37.3,yes,NaN,lipoma,NaN,NaN,M,Dom,"oncogene, fusion",T,HMGA2,NaN,NaN,"CCDS2516.1,CMKOR1,CXCR7,ENSG00000144476.5,GPR1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,ZNF429,zinc finger protein 429,353088.0,19:21505564-21538078,2,NaN,19p12,yes,NaN,GBM,NaN,NaN,O,NaN,NaN,Mis,NaN,NaN,NaN,"CCDS42537.1,ENSG00000197013.9,NM_001001415.2,N..."
749,ZNF479,zinc finger protein 479,90827.0,7:57119614-57139864,2,NaN,7p11.2,yes,NaN,"lung cancer, bladder carcinoma, prostate carci...",NaN,NaN,E,NaN,NaN,Mis,NaN,NaN,NaN,"CCDS43590.1,ENSG00000185177.12,KR19,NM_033273...."
750,ZNF521,zinc finger protein 521,25925.0,18:25061926-25352152,1,NaN,18q11.2,yes,NaN,ALL,NaN,NaN,L,Dom,"oncogene, fusion",T,PAX5,NaN,NaN,"CCDS32806.1,EHZF,ENSG00000198795.10,Evi3,NM_01..."
751,ZNRF3,zinc and ring finger 3,84133.0,22:28883592-29057487,2,NaN,22q12.1,yes,NaN,"colorectal cancer, adrenocortical carcinoma, g...",NaN,NaN,E,NaN,TSG,"N, F, Mis",NaN,NaN,NaN,"BK747E2.3,CCDS56225.1,ENSG00000183579.15,FLJ22..."


In [ ]:
import pandas as pd
import numpy as np

# First, ensure that the ID columns are the same type (usually integers or strings)
main_csv['Biomarker'] = main_csv['Biomarker'].astype(str)
cancer_gene_census['Gene Symbol'] = cancer_gene_census['Gene Symbol'].astype(str)

# Create a mapping from Entrez GeneId to Role in Cancer
role_mapping = cancer_gene_census.set_index('Gene Symbol')['Role in Cancer'].to_dict()

# Define a function to classify based on role
def classify_gene(entrez_id):
    role = role_mapping.get(entrez_id, None)
    if role:
        if 'TSG' in role:
            return 1
        elif 'oncogene' in role:
            return 0
    return np.nan  # Leave empty if no match or role is unknown

# Apply classification
main_csv['TSG_Label'] = main_csv['Biomarker'].apply(classify_gene)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,TSG_Label
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
biomarker_type = main_csv[['Biomarker','TSG_Label']]
biomarker_type

,Biomarker,TSG_Label
0,PTEN,1
1,PTEN,1
2,PTEN,1
3,PTEN,1
4,PTEN,1
...,...,...
144075,NRAS,0
144076,NRAS,0
144077,NRAS,0
144078,NRAS,0


In [ ]:
biomarker_type.to_csv('input_data/3_ML_outputs/feature_output/biomarker_type.csv', index=False)